# SPARK Pipeline — Google Colab (T4 / A10G / A100)
Exécuter les cellules dans l'ordre. Prévoir ~25-35 min pour un run complet (dont ~5 min de téléchargement du modèle Wan2.1 au premier run).

## 0. Vérification GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
# T4 (16 Go VRAM) ✓ suffisant pour Wan2.1 1.3B (~8 Go VRAM)
# A10G / A100 fonctionnent aussi — pour de meilleures performances

## 1. Cloner le repo

In [ ]:
import os
REPO_URL = 'https://github.com/Blockprod/SPARK.git'
PROJECT_DIR = '/content/SPARK'

# Clone SPARK
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} checkout -- .
    !git -C {PROJECT_DIR} pull

os.chdir(PROJECT_DIR)
print('Working dir:', os.getcwd())

## 2. Installer les dépendances

In [ ]:
# Install diffusers + Wan2.1 dependencies
!pip install -q diffusers transformers accelerate ftfy

# Install remaining SPARK dependencies
!pip install -q \
    pydantic-settings PyYAML python-dotenv httpx tenacity \
    google-genai \
    safetensors huggingface-hub \
    edge-tts \
    kokoro-onnx==0.4.9 onnxruntime-gpu soundfile librosa \
    ffmpeg-python pysubs2 opencv-python Pillow numpy \
    pytrends praw \
    google-api-python-client google-auth-httplib2 google-auth-oauthlib \
    APScheduler uvicorn sse-starlette jinja2 python-multipart \
    rich cryptography

!apt-get install -qq ffmpeg
print('Installation terminée (Wan2.1 1.3B via diffusers).')

## 3. Monter Google Drive et copier les secrets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_SECRETS = '/content/drive/MyDrive/SPARK_secrets'
import shutil, os

shutil.copy(f'{DRIVE_SECRETS}/.env', '/content/SPARK/.env')
os.makedirs('/content/SPARK/secrets', exist_ok=True)
shutil.copy(f'{DRIVE_SECRETS}/client_secret.json', '/content/SPARK/secrets/client_secret.json')
shutil.copy(f'{DRIVE_SECRETS}/youtube_token.json', '/content/SPARK/secrets/youtube_token.json')
os.makedirs('/content/SPARK/models', exist_ok=True)
shutil.copy(f'{DRIVE_SECRETS}/models/kokoro-v1.0.fp16.onnx', '/content/SPARK/models/kokoro-v1.0.fp16.onnx')
shutil.copy(f'{DRIVE_SECRETS}/models/voices-v1.0.bin', '/content/SPARK/models/voices-v1.0.bin')
print('Secrets copiés ✓')

## 4. Connexion HuggingFace (pour LTX-Video)

In [ ]:
from huggingface_hub import login
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print('HuggingFace connecté ✓')

In [ ]:
import psutil, shutil
ram_gb = psutil.virtual_memory().total / (1024**3)
disk = shutil.disk_usage('/')
disk_free_gb = disk.free / (1024**3)
print(f'RAM système : {ram_gb:.1f} Go')
print(f'Disque libre : {disk_free_gb:.1f} Go')
if ram_gb < 12:
    print('⚠️  RAM < 12 Go — risque OOM possible, activer High-RAM si disponible.')
else:
    print('✓ RAM suffisante pour Wan2.1 1.3B')
if disk_free_gb < 30:
    print('⚠️  Disque < 30 Go — le modèle Wan2.1 (~28 Go total) risque de ne pas tenir.')
else:
    print('✓ Espace disque OK')

## 5a. Modèle Wan2.1 1.3B

Le modèle (~28 Go total) est **téléchargé automatiquement** par Diffusers lors du premier `python pipeline.py`.

- Cache : `/root/.cache/huggingface/hub/` (réinitialisé à chaque session Colab)
- Durée : ~5 min sur le réseau Colab
- VRAM requise : ~8.2 Go → **compatible T4 gratuit (16 Go)**

Aucune action manuelle requise.

In [ ]:
# Wan2.1 1.3B est téléchargé automatiquement par diffusers au premier run.
# Pas de téléchargement manuel nécessaire.
print('Modèle Wan2.1 1.3B : téléchargement automatique par Diffusers au lancement.')
print('Taille         : ~28 Go total')
print('VRAM requise   : ~10 Go (cpu offload)')
print('Compatible T4  : oui avec use_cpu_offload=True')
print()
print('Le modèle sera mis en cache dans /root/.cache/huggingface/')
print('(réinitialisé à chaque session — re-téléchargement ~5 min par session)')

## 5b. Vérifier la config Wan2.1


In [ ]:
import yaml
config_path = '/content/SPARK/config.yaml'
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

# Force Wan2.1 config values (in case git pull brought old version)
cfg['video_generation']['provider'] = 'wan'
cfg['video_generation']['model_id'] = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
cfg['video_generation']['device'] = 'cuda'
cfg['video_generation']['num_frames'] = 81       # 4*20+1 = ~5 sec at 16 fps
cfg['video_generation']['num_inference_steps'] = 25
cfg['video_generation']['guidance_scale'] = 5.0
cfg['video_generation']['use_cpu_offload'] = True   # requis T4: text encoder UMT5-XXL ~10 Go VRAM
if 'scenes' not in cfg['video_generation']:
    cfg['video_generation']['scenes'] = {}
cfg['video_generation']['scenes']['max_scene_duration_sec'] = 5
cfg['pipeline']['target_width'] = 480
cfg['pipeline']['target_height'] = 832
cfg['pipeline']['target_fps'] = 16
# Subtitles — calibrated for 480x832 portrait
cfg['post_production']['subtitles']['style']['font_size'] = 42
cfg['post_production']['subtitles']['style']['bold'] = True
cfg['post_production']['subtitles']['style']['margin_v'] = 70
# Edge TTS configuration
if 'audio_generation' not in cfg:
    cfg['audio_generation'] = {}
cfg['audio_generation']['active_backend'] = 'edge_tts'
if 'edge_tts' not in cfg['audio_generation']:
    cfg['audio_generation']['edge_tts'] = {}
cfg['audio_generation']['edge_tts']['voice'] = 'fr-FR-DeniseNeural'
cfg['audio_generation']['edge_tts']['rate'] = '-5%'
cfg['audio_generation']['edge_tts']['pitch'] = '+0Hz'

with open(config_path, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False)

# Verification
with open(config_path, 'r') as f:
    check = yaml.safe_load(f)

v = check['video_generation']
p = check['pipeline']
print(f"  Provider         : {v['provider']}")
print(f"  Model            : {v.get('model_id', 'N/A')}")
print(f"  Resolution       : {p['target_width']}x{p['target_height']}")
print(f"  FPS              : {p['target_fps']}")
print(f"  Num frames       : {v.get('num_frames', 'N/A')} (4k+1 constraint)")
print(f"  Inference steps  : {v.get('num_inference_steps', 'N/A')}")
print(f"  CPU offload      : {v.get('use_cpu_offload', False)}")
print(f"  Sub font_size    : {check['post_production']['subtitles']['style']['font_size']}pt")
print(f"  Sub margin_v     : {check['post_production']['subtitles']['style']['margin_v']}")
print(f"  Audio backend    : {check['audio_generation']['active_backend']}")
print()
print('Config Wan2.1 vérifiée ✓ — prêt à lancer le pipeline')


## 6. Lancer le pipeline


In [ ]:
import os
TOPIC = "L'intelligence artificielle change le monde"
os.chdir('/content/SPARK')
!python pipeline.py --topic "{TOPIC}"


## 7. Télécharger le MP4 final


In [ ]:
import glob
from google.colab import files
renders = sorted(glob.glob('/content/SPARK/outputs/renders/*.mp4'))
if renders:
    final = renders[-1]
    print(f'Téléchargement: {final}')
    files.download(final)
else:
    print('Aucun fichier MP4 trouvé dans outputs/renders/')

# Optionnel: sauvegarder sur Drive
import shutil, os
DRIVE_OUTPUTS = '/content/drive/MyDrive/SPARK_outputs'
os.makedirs(DRIVE_OUTPUTS, exist_ok=True)
for mp4 in glob.glob('/content/SPARK/outputs/renders/*.mp4'):
    dest = os.path.join(DRIVE_OUTPUTS, os.path.basename(mp4))
    shutil.copy2(mp4, dest)
    print(f'Sauvegardé → {dest}')
